In [4]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from xgboost import XGBRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

# Setting plot style
sns.set(style="whitegrid")

In [5]:
train = pd.read_csv('train.csv')
test = pd.read_csv('test.csv')

In [6]:
# 1. Engineering the 'TotalSF' feature (Our 0.83 correlation winner)
train['TotalSF'] = train['TotalBsmtSF'] + train['1stFlrSF'] + train['2ndFlrSF']
test['TotalSF'] = test['TotalBsmtSF'] + test['1stFlrSF'] + test['2ndFlrSF']

# 2. Removing Outliers (Massive area, tiny price)
train = train.drop(train[(train['GrLivArea']>4000) & (train['SalePrice']<300000)].index)

In [7]:
# Handling missing values
# For categorical columns, NA usually means 'None'
cat_cols = ['PoolQC', 'MiscFeature', 'Alley', 'Fence', 'FireplaceQu', 'GarageType', 'GarageFinish', 'GarageQual', 'GarageCond', 'BsmtQual', 'BsmtCond', 'BsmtExposure', 'BsmtFinType1', 'BsmtFinType2', 'MasVnrType']
for col in cat_cols:
    train[col] = train[col].fillna('None')
    test[col] = test[col].fillna('None')

# Smart fill for LotFrontage based on neighborhood median
train["LotFrontage"] = train.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))
test["LotFrontage"] = test.groupby("Neighborhood")["LotFrontage"].transform(lambda x: x.fillna(x.median()))

# Fill numerical columns with 0
num_cols = ['GarageYrBlt', 'GarageArea', 'GarageCars', 'BsmtFinSF1', 'BsmtFinSF2', 'BsmtUnfSF','TotalBsmtSF', 'BsmtFullBath', 'BsmtHalfBath', 'MasVnrArea']
for col in num_cols:
    train[col] = train[col].fillna(0)
    test[col] = test[col].fillna(0)

In [8]:
# Label Encoding for quality features
qual_map = {'Ex': 5, 'Gd': 4, 'TA': 3, 'Fa': 2, 'Po': 1, 'None': 0}
qual_columns = ['ExterQual', 'ExterCond', 'BsmtQual', 'BsmtCond', 'HeatingQC', 'KitchenQual', 'FireplaceQu', 'GarageQual', 'GarageCond', 'PoolQC']

for col in qual_columns:
    train[col] = train[col].map(qual_map)
    test[col] = test[col].map(qual_map)

# One-Hot Encoding for remaining categorical data
train = pd.get_dummies(train)
test = pd.get_dummies(test)

# Align train and test columns
train_labels = np.log1p(train['SalePrice']) # Log Transformation
train = train.drop(['SalePrice', 'Id'], axis=1)
test_id = test['Id']
test = test.drop(['Id'], axis=1)

# Ensure both have same columns after dummies
train, test = train.align(test, join='left', axis=1, fill_value=0)

In [9]:
X_train, X_val, y_train, y_val = train_test_split(train, train_labels, test_size=0.2, random_state=42)

In [10]:
model = XGBRegressor(n_estimators=1000, learning_rate=0.05, max_depth=3, n_jobs=-1)
model.fit(X_train, y_train)

# Validation Score
predictions = model.predict(X_val)
rmse = np.sqrt(mean_squared_error(y_val, predictions))
print(f"Validation Log-RMSE: {rmse}")

Validation Log-RMSE: 0.11623670169479935


In [11]:
final_preds = np.expm1(model.predict(test)) # Un-doing the Log
submission = pd.DataFrame({'Id': test_id, 'SalePrice': final_preds})
submission.to_csv('submission.csv', index=False)